# 03 — Modelling

## About

**Purpose:** Train an XGBoost classifier to predict provider exclusion risk, handling severe class imbalance and tracking experiments in MLflow.<br>
**Author:** Ganapathy K<br>
**Date:** 2026-05-15<br>
**Notes:** Operates on the 500,000-row labelled dataset from 01. Target `excluded` is imbalanced at 0.237% — 1,183 excluded providers in 500,000, 237 of them in the test split.<br>
**Description:** Loads the labelled dataset, drops sparse and free-text columns, splits the rows, then target-encodes high-cardinality categoricals on the training rows only and one-hot encodes the rest (331 → 16 features). Trains XGBoost with `scale_pos_weight` to counter the imbalance, benchmarks it against two baselines and a soft-voting ensemble, logs to MLflow, states the decision threshold, and writes `serving/encoding_maps.json` for the serving app.

### Change Control

| Date       | Version | Author      | Changes         |
|------------|---------|-------------|-----------------|
| 2026-05-15 | 1.0     | Ganapathy K | Initial version |
| 2026-09-06 | 2.0     | Ganapathy K | Target encodings refitted on the training rows only — they had been fitted on all 500,000 rows before the split, so each test row's own label reached the feature it was scored on. Every metric below is regenerated and lower. Decision threshold stated and justified instead of inherited. Section 5.2's SMOTE figures flagged as unsupported. |

**These numbers now match `src/` exactly.** `python src/model.py` reproduces the recall, ROC-AUC and flagged count printed here, and section 3.5 writes the same `serving/encoding_maps.json` the deployed model was trained with — verified value for value. Notebook and deployed service can no longer disagree silently.

In [1]:
%load_ext autoreload
%autoreload 2

## 1. Setup
### 1.1 Imports

In [2]:
import json
import logging
from datetime import datetime
from pathlib import Path

import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score, f1_score, recall_score, precision_score, average_precision_score
import mlflow
import mlflow.xgboost

D:\Data Science\Visual Studio Code\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1.2 Configure logging

In [3]:
log_folder = Path("logs")
log_folder.mkdir(exist_ok=True)
log_filename = log_folder / f"run_{datetime.now().strftime('%Y-%m-%d')}.log"

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    handlers=[
        logging.FileHandler(log_filename, encoding='utf-8'),
        logging.StreamHandler(),
    ],
    force=True,
)
logger = logging.getLogger(__name__)

### 1.3 Config

In [4]:
processed_file_path = Path("D:/Data Science/Visual Studio Code/healthcare-provider-exclusion-risk/data/processed/labelled_dataset.parquet")

## 2. Load Data

In [5]:
# Load labelled dataset from parquet
providers_raw_df = pd.read_parquet(processed_file_path)
providers_df = providers_raw_df.copy()
logger.info(f"Loaded labelled dataset — shape: {providers_df.shape}")

2026-09-06 10:13:24,928 | INFO | Loaded labelled dataset — shape: (500000, 331)


## 3. Preprocessing
### 3.1 Drop Sparse Columns

In [6]:
# Null values treatment — drop columns with >30% nulls
null_percentages = providers_df.isnull().mean().sort_values(ascending=False) * 100
high_null_columns = null_percentages[null_percentages > 30].index
providers_df.drop(columns=high_null_columns, inplace=True)
logger.info(f"After dropping >30%-null columns — shape: {providers_df.shape}")

2026-09-06 10:13:26,187 | INFO | After dropping >30%-null columns — shape: (500000, 26)


### 3.2 Drop Identifier and Free-Text Columns

In [7]:
# Drop identifier and free-text columns
columns_to_drop = [
    'NPI',
    'Provider Last Name (Legal Name)',
    'Provider First Name',
    'Provider Credential Text',
    'Provider First Line Business Mailing Address',
    'Provider First Line Business Practice Location Address',
    'Provider Business Practice Location Address Telephone Number'
]
providers_df.drop(columns=columns_to_drop, inplace=True, errors='ignore')
logger.info(f"After dropping identifier/free-text columns — shape: {providers_df.shape}")

2026-09-06 10:13:26,304 | INFO | After dropping identifier/free-text columns — shape: (500000, 19)


### 3.3 Dtype Fixes

In [8]:
# Dtype fixes — date columns and Entity Type Code
providers_df['Provider Enumeration Year'] = pd.to_datetime(providers_df['Provider Enumeration Date']).dt.year.astype('Int64')
providers_df['Last Update Year'] = pd.to_datetime(providers_df['Last Update Date']).dt.year.astype('Int64')
providers_df['Entity Type Code'] = providers_df['Entity Type Code'].astype('category')
providers_df.drop(columns=['Provider Enumeration Date', 'Last Update Date'], inplace=True)
print(providers_df.dtypes)

Entity Type Code                                                              category
Provider Business Mailing Address City Name                                     object
Provider Business Mailing Address State Name                                    object
Provider Business Mailing Address Postal Code                                   object
Provider Business Mailing Address Country Code (If outside U.S.)                object
Provider Business Mailing Address Telephone Number                             float64
Provider Business Practice Location Address City Name                           object
Provider Business Practice Location Address State Name                          object
Provider Business Practice Location Address Postal Code                         object
Provider Business Practice Location Address Country Code (If outside U.S.)      object
Provider Sex Code                                                               object
Healthcare Provider Taxonomy Code_1        

### 3.4 Split the Rows, Then Encode Categorical Columns

In [9]:
# Encode categorical columns
object_column_cardinality = providers_df.select_dtypes(include='object').nunique().sort_values(ascending=False)
high_cardinality_columns_to_drop = object_column_cardinality[object_column_cardinality > 1000].index
providers_df.drop(columns=high_cardinality_columns_to_drop, inplace=True)

# Drop near-zero variance columns — 99%+ US providers, country code adds no signal
near_zero_variance_columns_to_drop = [
    'Provider Business Mailing Address Country Code (If outside U.S.)',
    'Provider Business Practice Location Address Country Code (If outside U.S.)'
]

providers_df.drop(columns=near_zero_variance_columns_to_drop, inplace=True, errors='ignore')

# Split the ROWS here, before anything is fitted on the target. The four encodings below
# replace a category with its mean exclusion rate; fitted on all 500,000 rows, a test row's
# own label goes into the number that same row is later scored on. Fitting them on the
# training rows only is the fix. The split is on the index, so 4.3 reuses these exact rows.
train_index, test_index = train_test_split(
    providers_df.index, test_size=0.2, random_state=42, stratify=providers_df['excluded'])
train_rows_df = providers_df.loc[train_index]

# Target encode high-cardinality categorical columns — replace category with mean exclusion rate
target_encoded_columns = [
    'Healthcare Provider Taxonomy Code_1',
    'Provider Business Mailing Address State Name',
    'Provider Business Practice Location Address State Name',
    'Provider License Number State Code_1',
]

# A category present only in the test rows stays unmapped and becomes NaN, which the median
# fill in 4.2 handles. That is the honest outcome — at serving time a taxonomy code never
# seen in training has no history either.
encoding_maps = {}
for column_name in target_encoded_columns:
    encoding_map = train_rows_df.groupby(column_name)['excluded'].mean()
    encoding_maps[column_name] = encoding_map
    providers_df[column_name] = providers_df[column_name].map(encoding_map)

providers_df = pd.get_dummies(providers_df, columns=['Provider Sex Code', 'Healthcare Provider Primary Taxonomy Switch_1', 'Is Sole Proprietor'])

logger.info(f"After encoding — shape: {providers_df.shape}")
logger.info(f"Encodings fitted on {len(train_index):,} training rows only")

2026-09-06 10:13:27,213 | INFO | After encoding — shape: (500000, 17)


2026-09-06 10:13:27,214 | INFO | Encodings fitted on 400,000 training rows only


### 3.5 Save Encoding Maps

In [10]:
# Save target encoding maps for serving — these MUST ship with the model trained on them.
# serving/app.py looks up every categorical value here, so a model trained on train-only means
# paired with a file of full-data means would be scored on numbers it never saw, silently.
encoding_maps_path = Path("../serving/encoding_maps.json")
with open(encoding_maps_path, "w") as file:
    json.dump({name: mapping.to_dict() for name, mapping in encoding_maps.items()}, file)

logger.info(f"Saved encoding maps to: {encoding_maps_path.resolve()}")
for column_name, mapping in encoding_maps.items():
    logger.info(f"  {column_name}: {len(mapping)} categories")

2026-09-06 10:13:27,272 | INFO | Saved encoding maps to: D:\Data Science\Visual Studio Code\healthcare-provider-exclusion-risk\serving\encoding_maps.json


2026-09-06 10:13:27,273 | INFO |   Healthcare Provider Taxonomy Code_1: 745 categories


2026-09-06 10:13:27,273 | INFO |   Provider Business Mailing Address State Name: 121 categories


2026-09-06 10:13:27,273 | INFO |   Provider Business Practice Location Address State Name: 140 categories


2026-09-06 10:13:27,274 | INFO |   Provider License Number State Code_1: 60 categories


## 4. Train/Test Split
### 4.1 Define Features and Target

In [11]:
# Define features and target variable
X = providers_df.drop(columns=['excluded'])
y = providers_df['excluded']
logger.info(f"Features X: {X.shape} | target y: {y.shape}")

2026-09-06 10:13:27,331 | INFO | Features X: (500000, 16) | target y: (500000,)


### 4.2 Handle Remaining Nulls

In [12]:
# Handle nulls — fill with median after target encoding
# The median is taken over all 500,000 rows, not the training rows alone. That is a smaller
# version of the leak fixed in 3.4, left as-is deliberately: src/model.py does the same, and
# changing it here would fork this notebook from the model actually deployed. At 500,000 rows
# and 0.237% prevalence the two medians agree; it is documented rather than hidden.
X['Entity Type Code'] = X['Entity Type Code'].astype('float')
X = X.fillna(X.median(numeric_only=True))
logger.info(f"Remaining nulls in X: {X.isnull().sum().sum()}")

2026-09-06 10:13:27,471 | INFO | Remaining nulls in X: 0


### 4.3 Stratified Split

In [13]:
# Train-test split — reuse the rows split in 3.4, before the encodings were fitted.
# Same seed, same stratification, so this is the same split, just taken earlier.
X_train, X_test = X.loc[train_index], X.loc[test_index]
y_train, y_test = y.loc[train_index], y.loc[test_index]
logger.info(f"Train: {X_train.shape} | Test: {X_test.shape}")
logger.info(f"Excluded providers — train: {int(y_train.sum())} | test: {int(y_test.sum())}")

2026-09-06 10:13:27,573 | INFO | Train: (400000, 16) | Test: (100000, 16)


2026-09-06 10:13:27,573 | INFO | Excluded providers — train: 946 | test: 237


## 5. Modelling
### 5.1 XGBoost with scale_pos_weight

In [14]:
# XGBoost Run 2 — scale_pos_weight instead of SMOTE
xgboost_model_spw = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='aucpr',
    scale_pos_weight=422
)

with mlflow.start_run(run_name="XGBoost with scale_pos_weight"):
    mlflow.log_params({
        'n_estimators': 100,
        'max_depth': 6,
        'learning_rate': 0.1,
        'imbalance_method': 'scale_pos_weight',
        'scale_pos_weight': 422,
        'eval_metric': 'aucpr'
    })

    xgboost_model_spw.fit(X_train, y_train)
    mlflow.xgboost.log_model(xgboost_model_spw, name="xgboost_model")

    y_pred_spw = xgboost_model_spw.predict(X_test)
    y_proba_spw = xgboost_model_spw.predict_proba(X_test)[:, 1]

    mlflow.log_metrics({
        'recall': recall_score(y_test, y_pred_spw),
        'precision': precision_score(y_test, y_pred_spw),
        'f1': f1_score(y_test, y_pred_spw),
        'roc_auc': roc_auc_score(y_test, y_proba_spw),
        'average_precision': average_precision_score(y_test, y_proba_spw)
    })

    logger.info("MLflow run complete — params + metrics logged")
    logger.info(f"Recall: {recall_score(y_test, y_pred_spw):.4f} | ROC-AUC: {roc_auc_score(y_test, y_proba_spw):.4f} | Average precision: {average_precision_score(y_test, y_proba_spw):.4f}")

print(classification_report(y_test, y_pred_spw))

2026-09-06 10:13:33,607 | INFO | MLflow run complete — params + metrics logged


2026-09-06 10:13:33,646 | INFO | Recall: 0.6624 | ROC-AUC: 0.7536 | Average precision: 0.0084


              precision    recall  f1-score   support

           0       1.00      0.74      0.85     99763
           1       0.01      0.66      0.01       237

    accuracy                           0.74    100000
   macro avg       0.50      0.70      0.43    100000
weighted avg       1.00      0.74      0.85    100000



### 5.2 Alternative tried — SMOTE

Before settling on `scale_pos_weight`, the intention was to oversample the minority class with SMOTE (Synthetic Minority Over-sampling Technique) on the training set.

| Run | Approach | Recall | ROC-AUC | Precision |
|-----|----------|--------|---------|-----------|
| 1 | SMOTE oversampling | 0.177 | 0.765 | ~0.00 |
| 2 | `scale_pos_weight=422` | **0.662** | **0.754** | 0.01 |

⚠️ **Row 1 is not evidence and should not be quoted as if it were.** There is no SMOTE cell in this notebook and no SMOTE run in `mlruns/`, and 0.177 / 0.765 are exactly the figures `src/baseline.py` measured for the shipped **unweighted** XGBoost model — the wrong-model defect described in the README, not a SMOTE result. Either the run was never made or its numbers were overwritten by that measurement. Until it is re-run, treat row 1 as unverified.

The argument for `scale_pos_weight` over SMOTE stands on its own and does not need row 1: at 0.237% prevalence, synthetic minority points are interpolated between real minority points that already sit very close together, so they add volume rather than signal. `scale_pos_weight` reweights the loss instead of inventing rows, costs nothing at training time, and is the same lever the decision threshold in 6.1 turns out to pull.

Row 2 is logged to MLflow. Note that `mlruns/` is gitignored, so the log is local to whoever ran the notebook — the reproducible record is this notebook's own output.

### 5.3 Baseline Comparison

XGBoost was the production choice, but a single model in isolation says nothing about *lift* — how much that choice actually buys. This section benchmarks it against two simpler baselines, trained on the same split with the same imbalance handling (`class_weight='balanced'` — the LogReg / Random Forest equivalent of `scale_pos_weight`):

- **Logistic Regression** — linear baseline; the floor any tree model must clear. Wrapped in a `StandardScaler` pipeline because LogReg is scale-sensitive.
- **Random Forest** — bagged-tree baseline; isolates the gain that comes from boosting specifically versus plain tree ensembling.

Recall is the metric to compare on — this is a screening model, so missing an excluded provider is the costly error.

In [15]:
# Baseline models — same split, same imbalance handling (class_weight='balanced')
logistic_regression_model = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)),
])
random_forest_model = RandomForestClassifier(
    n_estimators=100, max_depth=6, class_weight='balanced', random_state=42, n_jobs=-1
)

baseline_models = {
    'Logistic Regression': logistic_regression_model,
    'Random Forest': random_forest_model,
}

model_comparison_rows = []
for model_name, model in baseline_models.items():
    with mlflow.start_run(run_name=model_name):
        model.fit(X_train, y_train)
        y_pred_baseline = model.predict(X_test)
        y_proba_baseline = model.predict_proba(X_test)[:, 1]
        baseline_metrics = {
            'recall': recall_score(y_test, y_pred_baseline),
            'precision': precision_score(y_test, y_pred_baseline),
            'f1': f1_score(y_test, y_pred_baseline),
            'roc_auc': roc_auc_score(y_test, y_proba_baseline),
        }
        mlflow.log_param('imbalance_method', 'class_weight=balanced')
        mlflow.log_metrics(baseline_metrics)
        model_comparison_rows.append({'model': model_name, **baseline_metrics})
        logger.info(f"{model_name} — Recall: {baseline_metrics['recall']:.4f} | ROC-AUC: {baseline_metrics['roc_auc']:.4f}")

# XGBoost numbers from the scale_pos_weight run in 5.1
model_comparison_rows.append({
    'model': 'XGBoost (scale_pos_weight)',
    'recall': recall_score(y_test, y_pred_spw),
    'precision': precision_score(y_test, y_pred_spw),
    'f1': f1_score(y_test, y_pred_spw),
    'roc_auc': roc_auc_score(y_test, y_proba_spw),
})

model_comparison_df = pd.DataFrame(model_comparison_rows).sort_values('recall', ascending=False).reset_index(drop=True)
print(model_comparison_df)

2026-09-06 10:13:34,334 | INFO | Logistic Regression — Recall: 0.6962 | ROC-AUC: 0.7706


2026-09-06 10:13:36,651 | INFO | Random Forest — Recall: 0.6878 | ROC-AUC: 0.7682


                        model    recall  precision        f1   roc_auc
0         Logistic Regression  0.696203   0.005402  0.010721  0.770560
1               Random Forest  0.687764   0.005763  0.011430  0.768187
2  XGBoost (scale_pos_weight)  0.662447   0.006004  0.011900  0.753602


### 5.4 Three-Model Ensemble (measured-lift check)

A soft-voting ensemble averages the predicted probabilities of all three models. The goal here is not to ship the ensemble — it is to *measure* whether combining models beats XGBoost alone:

- If the ensemble's recall is within noise of XGBoost, that is evidence the single model is the right production choice — simpler, faster, one artifact to serve.
- If it lifts recall materially, that is a documented, numeric reason to revisit.

Either way, the notebook now states the lift explicitly rather than assuming XGBoost is best.

In [16]:
# Soft-voting ensemble — averages the predicted probabilities of all three models
xgboost_model_for_ensemble = XGBClassifier(
    n_estimators=100, max_depth=6, learning_rate=0.1,
    random_state=42, eval_metric='aucpr', scale_pos_weight=422,
)

ensemble_model = VotingClassifier(
    estimators=[
        ('logistic_regression', logistic_regression_model),
        ('random_forest', random_forest_model),
        ('xgboost', xgboost_model_for_ensemble),
    ],
    voting='soft',
)

with mlflow.start_run(run_name="Soft-Voting Ensemble"):
    ensemble_model.fit(X_train, y_train)
    y_pred_ensemble = ensemble_model.predict(X_test)
    y_proba_ensemble = ensemble_model.predict_proba(X_test)[:, 1]
    ensemble_metrics = {
        'recall': recall_score(y_test, y_pred_ensemble),
        'precision': precision_score(y_test, y_pred_ensemble),
        'f1': f1_score(y_test, y_pred_ensemble),
        'roc_auc': roc_auc_score(y_test, y_proba_ensemble),
    }
    mlflow.log_metrics(ensemble_metrics)
    logger.info(f"Ensemble — Recall: {ensemble_metrics['recall']:.4f} | ROC-AUC: {ensemble_metrics['roc_auc']:.4f}")

model_comparison_with_ensemble_df = pd.concat([
    model_comparison_df,
    pd.DataFrame([{'model': 'Soft-Voting Ensemble', **ensemble_metrics}]),
], ignore_index=True).sort_values('recall', ascending=False).reset_index(drop=True)

best_single_model_recall = model_comparison_df['recall'].max()
ensemble_recall_lift = ensemble_metrics['recall'] - best_single_model_recall
logger.info(f"Ensemble recall vs best single model: {ensemble_recall_lift:+.4f}")
print(model_comparison_with_ensemble_df)

2026-09-06 10:13:40,056 | INFO | Ensemble — Recall: 0.6709 | ROC-AUC: 0.7780


2026-09-06 10:13:40,065 | INFO | Ensemble recall vs best single model: -0.0253


                        model    recall  precision        f1   roc_auc
0         Logistic Regression  0.696203   0.005402  0.010721  0.770560
1               Random Forest  0.687764   0.005763  0.011430  0.768187
2        Soft-Voting Ensemble  0.670886   0.005867  0.011632  0.778006
3  XGBoost (scale_pos_weight)  0.662447   0.006004  0.011900  0.753602


### 5.5 What the comparison shows

| Model | Recall | ROC-AUC |
|-------|--------|---------|
| Logistic Regression | 0.696 | 0.771 |
| Random Forest | 0.688 | 0.768 |
| Soft-Voting Ensemble | 0.671 | 0.778 |
| XGBoost (`scale_pos_weight`) | 0.662 | 0.754 |

Run cold, the result is worth stating plainly: **XGBoost is not the top model on recall here.** With identical imbalance handling, all four land inside a narrow 0.66–0.70 recall band and ROC-AUC inside 0.75–0.78. The ensemble buys the best ROC-AUC (0.778) but its recall is 0.025 *below* the best single model. Precision and F1 are all around 0.006–0.012 at this prevalence — not decision-useful.

The real lever is the **imbalance handling, not the model family**: `class_weight='balanced'` / `scale_pos_weight` is what carries every row of this table into the 0.66–0.70 band. XGBoost is kept as the served model not because it tops the table but for engineering reasons — native `scale_pos_weight`, first-class MLflow logging, and a single fast artifact. Logistic Regression is 0.034 recall ahead and is a genuine, cheaper alternative; the reason not to switch is that at 0.006 precision none of these differences survive contact with a review queue, and swapping the served model buys nothing measurable.

The value of this section is that the choice is now *documented and defensible* rather than assumed.

## 6. Evaluation
### 6.1 The Decision Threshold

In [17]:
# The cut-off is a choice, so it gets made rather than inherited. XGBoost predicts at 0.5 by
# default; 0.5 is the right cut-off only when the two classes are equally common and the two
# mistakes cost the same, and neither holds here.
#
# It is CHOSEN in src/threshold.py, on out-of-fold predictions over the training rows only —
# picking it on this test split would repeat, at the decision layer, exactly the leakage the
# encoding maps had at the feature layer. That fit returns 0.37 on the training rows and 0.45
# on the test rows, and neither beats 0.5 out of sample.
#
# The reason is that scale_pos_weight=422 has already made the decision: training tells the
# model one missed exclusion costs 422 false flags, and for a model weighted that way 0.5 IS
# the cost-optimal cut-off. Weighting and moving the cut-off are one lever pulled in two
# places. The sweep below is what that choice buys on this test split.
RISK_THRESHOLD = 0.5
COST_RATIO = 422

actual = y_test.to_numpy()
excluded_in_test = int(actual.sum())

sweep_rows = []
for candidate in [0.1, 0.2, 0.3, 0.37, 0.45, 0.5, 0.6, 0.7, 0.8, 0.9]:
    flagged = y_proba_spw >= candidate
    caught = int(flagged[actual == 1].sum())
    false_alarms = int(flagged.sum()) - caught
    missed = excluded_in_test - caught
    sweep_rows.append({
        'threshold': candidate,
        'flagged': int(flagged.sum()),
        'caught': caught,
        'missed': missed,
        'recall': round(caught / excluded_in_test, 4),
        'cost': COST_RATIO * missed + false_alarms,
    })

threshold_sweep_df = pd.DataFrame(sweep_rows)
print(threshold_sweep_df.to_string(index=False))

served = y_proba_spw >= RISK_THRESHOLD
logger.info(f"Serving at {RISK_THRESHOLD} — flags {int(served.sum()):,} providers "
            f"to catch {int(served[actual == 1].sum())} of {excluded_in_test} excluded")

2026-09-06 10:13:40,142 | INFO | Serving at 0.5 — flags 26,149 providers to catch 157 of 237 excluded


 threshold  flagged  caught  missed  recall  cost
      0.10    57519     212      25  0.8945 67857
      0.20    46262     196      41  0.8270 63368
      0.30    38085     181      56  0.7637 61536
      0.37    33477     172      65  0.7257 60735
      0.45    28842     166      71  0.7004 58638
      0.50    26149     157      80  0.6624 59752
      0.60    20974     135     102  0.5696 63883
      0.70     7519      72     165  0.3038 77077
      0.80     3007      41     196  0.1730 85678
      0.90      269       5     232  0.0211 98168


**What the sweep says.** Cost is `422 × missed + false alarms`, the same 422 the model was trained with.

- **0.45 costs less than 0.5 on this split** — 58,638 against 59,752, about 1.9% — but 0.45 is the number `src/threshold.py` gets when it fits on the *test* rows. Choosing it because it wins on the split it was chosen from is the same mistake the encoding maps made, one layer down. Fitted honestly, out-of-fold on the training rows, the answer is 0.37 — and 0.37 costs **60,735**, more than 0.5.
- The whole surface between 0.37 and 0.5 is flat to about 4%, so no cut-off in that range is meaningfully better than another.
- The ends are the useful part of the table: at 0.1 the queue is 57,519 providers long to catch 212, at 0.9 it is 269 long and catches 5. **Neither of those is the model's decision — they are the business's.**

So 0.5 stays, now for a reason rather than by inheritance.

### 6.2 Top-20 Risk Scores

In [18]:
# Add predicted probability to test set
X_test_results_df = X_test.copy()
X_test_results_df['excluded_actual'] = y_test.values
X_test_results_df['exclusion_risk_score'] = y_proba_spw

# Show top 20 highest-risk providers
top_20_risk_df = X_test_results_df.sort_values('exclusion_risk_score', ascending=False).head(20)
print(top_20_risk_df[['exclusion_risk_score', 'excluded_actual']])

        exclusion_risk_score  excluded_actual
72028               0.975570                0
86201               0.974140                0
428643              0.973946                0
290112              0.973760                0
490557              0.971970                0
491717              0.971970                0
263880              0.970032                0
191999              0.968888                0
421446              0.967520                0
173553              0.966666                0
159366              0.965172                0
462472              0.964515                0
396755              0.963998                0
113414              0.962971                0
51872               0.962365                0
451826              0.961354                0
229889              0.960789                0
78677               0.960551                0
55358               0.959797                0
101634              0.959453                0


## 7. Summary

**Pitch:** Trained an XGBoost classifier to flag providers at risk of OIG exclusion, handling 0.237% class imbalance with `scale_pos_weight`, benchmarked it against baselines, found and removed a target-leak in the feature encodings, and chose the decision threshold instead of inheriting it.

### Approach
- Preprocessing: dropped >30%-null and free-text columns (331 → 16 features), target-encoded high-cardinality categoricals, one-hot encoded the rest
- **Split first, encode second** — the four target encodings are fitted on the 400,000 training rows only. Fitted on all 500,000, each test row's own label reached the feature it was later scored on
- Imbalance: `scale_pos_weight=422`, the negative/positive ratio
- Baselines: benchmarked against Logistic Regression and Random Forest (both `class_weight='balanced'`), plus a soft-voting ensemble, to quantify lift instead of assuming XGBoost wins
- Tracking: every run logged to MLflow with params + metrics

### Results (`scale_pos_weight` run, leak-free)
| | value |
|---|---|
| excluded providers caught | **157 of 237** |
| recall | **0.662** |
| ROC-AUC | **0.754** |
| average precision | 0.0084 |
| providers flagged | **26,149** of 100,000 |
| decision threshold | **0.5**, chosen in `src/threshold.py` |

Accuracy is not reported because it is meaningless here — predicting "not excluded" for every provider scores 99.76%. Recall is what this model exists to produce.

**The cost, stated plainly: 26,149 providers go into a review queue to catch 157 of the 237.** Whether that queue is affordable is the business's number, not this notebook's — 422 is the assumption that sets its length, and 6.1 shows what other cut-offs would cost.

### What removing the leak cost
| | leaky encodings | training-rows-only |
|---|---|---|
| excluded caught (of 237) | 165 | **157** |
| recall | 0.696 | **0.662** |
| ROC-AUC | 0.800 | **0.754** |

Every number got worse, which is the point — the earlier ones were partly measuring their own answer sheet. The leak was worth about 8 of the 157, and ROC-AUC had been overstated by roughly 6%.

### Serving handoff
- `serving/encoding_maps.json` written from the **training-rows-only** maps, so the FastAPI app encodes with the same numbers the model was trained on
- The threshold is stamped onto `serving/model.ubj` by `src/threshold.py --write` and read back at startup, so the served cut-off cannot drift from the chosen one